In [2]:
import requests
import pandas as pd
from datetime import datetime, timezone
import time
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from matplotlib.cm import get_cmap
from matplotlib.patches import Patch
import ast
import folium
from folium.plugins import MarkerCluster
import reverse_geocoder as rg
import re
import pycountry
import os
import numpy as np
import geopandas as gpd
import fiona
import sys
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import ruptures as rpt
from haversine import haversine
import functions as own
from timezonefinder import TimezoneFinder
import zoneinfo

Now, we create plots for every country and every first order region with daily observation counts and running monthly means.

In [3]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])

df["date"] = pd.to_datetime(df["date"])

start_date = pd.to_datetime("2017-02-01")
end_date   = pd.to_datetime("2026-01-31")

def plot_country(country, split_by_year=True):
    base_path = f"../Products/Timelines_by_Country/{country}"
    os.makedirs(base_path, exist_ok=True)

    df_filtered = df[df["Country"] == country].copy()
    _plot(df_filtered, base_path, title=country, filename_prefix=country, split_by_year=split_by_year)

def plot_region(country, region, split_by_year=True):
    
    base_path = f"../Products/Timelines_by_Country/{country}/regions/{region}"
    os.makedirs(base_path, exist_ok=True)

    df_filtered = df[(df["Country"] == country) & (df["Region"] == region)].copy()
    _plot(df_filtered, base_path, title=f"{country} - {region}", filename_prefix=f"{country}_{region}", split_by_year=split_by_year)

def _plot(df_filtered, base_path, title, filename_prefix, split_by_year=True):
    # internal helping function
    
    full_range = pd.date_range(start=start_date, end=end_date, freq="D")

    daily = (
        df_filtered
        .groupby("date")
        .size()
        .rename("daily_observations")
        .reindex(full_range, fill_value=0)
    )
    rolling_30 = daily.rolling(window=30, min_periods=1).mean()
    data = pd.DataFrame({"daily": daily, "rolling_30": rolling_30})

    # Full plot
    _save_plot(data.index, data["daily"], data["rolling_30"], 
               title=title, 
               path=f"{base_path}/timeline_{filename_prefix}_full.png")

    if split_by_year:
        for year in range(2017, 2027):
            start = pd.to_datetime(f"{year}-01-01")
            end   = pd.to_datetime(f"{year}-12-31")
            if year == 2017: start = pd.to_datetime("2017-02-01")
            if year == 2026: end   = pd.to_datetime("2026-01-31")

            df_year = data.loc[start:end]
            if df_year["daily"].sum() == 0:
                continue

            _save_plot(df_year.index, df_year["daily"], df_year["rolling_30"],
                       title=f"{title} - {year}",
                       path=f"{base_path}/timeline_{filename_prefix}_{year}.png")

def _save_plot(index, daily, rolling, title, path):
    
    plt.figure(figsize=(16, 8))
    plt.plot(index, daily)
    plt.plot(index, rolling)
    plt.title(title, fontsize=15)
    plt.xlabel("Date")
    plt.ylabel("Observations")
    plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.6)
    plt.legend(["Daily observations", "Running 30-day mean"], loc="upper right")
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_3488\3695409245.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 4

In [4]:
for country in df["Country"].unique():
    plot_country(country)

for (country, region) in df[["Country", "Region"]].drop_duplicates().values:
    plot_region(country, region)

Make dataframes of daily observations in each and every first order subregion and in every country of the world, in order to later detect events in the data (=unusual patterns with spikes in observations).

In [5]:
df = pd.read_csv("../CWData_clean7.csv")
df["date"] = pd.to_datetime(df["date"])

daily_region = (
    df
    .groupby(["Country", "Region", "date"])
    .size()
    .rename("daily_obs")
    .reset_index()
)

daily_country = (
    df.groupby(["Country", "date"])
    .size()
    .rename("daily_obs")
    .reset_index()
)

C:\Users\yanni\AppData\Local\Temp\ipykernel_3488\423652575.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 42

calculate z-score (using daily observations, 60-day mean observation counts and 60-day standard deviation) for every day for every region and every country.

In [6]:
# uses daily_country and daily_region

full_range = pd.date_range("2017-02-01", "2026-01-31", freq="D")

all_regions = []

for (country, region), group in daily_region.groupby(["Country", "Region"]):

    d = group.set_index("date").sort_index()
    d = d.reindex(full_range)
    d["daily_obs"] = d["daily_obs"].fillna(0)
    d["Country"] = country
    d["Region"] = region
    d["mean_60"] = d["daily_obs"].rolling(60).mean()
    d["std_60"]  = d["daily_obs"].rolling(60).std()
    d["z_score"] = (d["daily_obs"] - d["mean_60"]) / d["std_60"]

    d = d.reset_index().rename(columns={"index": "date"})

    all_regions.append(d)

region_events_df = pd.concat(all_regions, ignore_index=True)

all_countries = []

for country, group in daily_country.groupby("Country"):

    d = group.set_index("date").sort_index()
    d = d.reindex(full_range)
    d["daily_obs"] = d["daily_obs"].fillna(0)
    d["Country"] = country
    d["mean_60"] = d["daily_obs"].rolling(60).mean()
    d["std_60"]  = d["daily_obs"].rolling(60).std()
    d["z_score"] = (d["daily_obs"] - d["mean_60"]) / d["std_60"]
    
    d = d.reset_index().rename(columns={"index": "date"})
    
    all_countries.append(d)

country_events_df = pd.concat(all_countries, ignore_index=True)

Function that detects events

In [7]:
def detect_events(df_input, group_col, z_threshold=2, block_length=3): # everything above threshold counts as unusual and if more than x days are above it, it's an event
    
    df_input = df_input.copy()
    df_input["is_event_day"] = df_input["z_score"] > z_threshold

    df_input["rolling_sum"] = (
        df_input
        .groupby(group_col)["is_event_day"]
        .transform(lambda x: x.rolling(block_length).sum())
    )

    df_input["event_block"] = (
        df_input
        .groupby(group_col)["rolling_sum"]
        .transform(lambda x: pd.concat(
            [x.shift(-i) for i in range(block_length)], axis=1
        ).max(axis=1) >= block_length)
    )

    periods = []

    for group_val in df_input[group_col].unique():

        d = df_input[df_input[group_col] == group_val].copy()
        country = d["Country"].iloc[0]

        d["event_shift"] = d["event_block"].shift(1, fill_value=False)

        starts = d[(d["event_block"] == True) & (d["event_shift"] == False)]
        ends   = d[(d["event_block"] == False) & (d["event_shift"] == True)]

        start_dates = starts["date"].values
        end_dates   = ends["date"].values

        if len(start_dates) > len(end_dates):
            end_dates = list(end_dates) + [d["date"].iloc[-1]]

        for s, e in zip(start_dates, end_dates):
            periods.append({
                "Country":    country,
                "Region":     group_val if group_col == "Region" else None,
                "level":      group_col,  # "Region" or "Country"
                "start_date": s,
                "end_date":   pd.to_datetime(e) - pd.Timedelta(days=1),
                "event_length (days)": (pd.to_datetime(e) - pd.to_datetime(s)).days
            })

    return pd.DataFrame(periods)

In [8]:
# uses country_events_df and region_events_df

events_list = []
events_list.append(detect_events(country_events_df, "Country", block_length=3))
events_list.append(detect_events(region_events_df, "Region", block_length=3))

pd.set_option("display.max_rows", None)
events_table = pd.concat(events_list, ignore_index=True)
events_table

,Country,Region,level,start_date,end_date,event_length (days)
0,Argentina,None,Country,2020-11-13,2020-11-15,3
1,Australia,None,Country,2021-05-31,2021-06-03,4
2,Australia,None,Country,2021-06-08,2021-06-11,4
3,Austria,None,Country,2018-01-10,2018-01-12,3
4,Austria,None,Country,2018-01-15,2018-01-17,3
5,Austria,None,Country,2021-01-31,2021-02-02,3
6,Brazil,None,Country,2023-10-02,2023-10-05,4
7,Brazil,None,Country,2024-10-22,2024-10-24,3
8,Canada,None,Country,2023-04-05,2023-04-07,3
9,Canada,None,Country,2025-03-07,2025-03-09,3


Change-Point-Detection (C. Truong, L. Oudre, N. Vayatis. Selective review of offline change point detection methods. Signal Processing, 167:107299, 2020.)

In [9]:
def detect_change_points(df_input, group_col, pen, min_size):

    change_points_list = []

    for group_val in df_input[group_col].unique():

        d = df_input[df_input[group_col] == group_val].copy()
        
        # if region has less than five total observations, skip it
        if d["daily_obs"].max() < 5:
            continue
        country = d["Country"].iloc[0]

        # remove time before first activity in region
        #first_activity = d[d["daily_obs"] > 0]["date"]
        #if len(first_activity) == 0:
            #continue

        #d = d[d["date"] >= first_activity.min()]

        # weekly aggregation
        d_weekly = (
            d.set_index("date")
             .resample("W")["daily_obs"]
             .sum()
             .reset_index()
        )

        d_weekly["cum_obs"] = d_weekly["daily_obs"].cumsum()

        d_weekly["growth"] = d_weekly["daily_obs"].pct_change().fillna(0)

        signal = d_weekly["daily_obs"].values

        if np.sum(signal) == 0:
            continue

        # Model
        model = rpt.Pelt(model="l2", min_size=min_size).fit(signal)

        if len(signal) < 2 * min_size:
            continue

        # calculate breakpoints
        breakpoints = model.predict(pen=pen)

        # predict() returns indices
        # last point is always length (--> ignore that one)
        for bp in breakpoints[:-1]:

            change_points_list.append({
                "Country": country,
                "Region": group_val if group_col == "Region" else None,
                "level": group_col,
                "change_point_date": d_weekly["date"].iloc[bp]
            })

    return pd.DataFrame(change_points_list)

In [10]:
# uses country_events_df and region_events_df

cp_list = []
cp_list.append(detect_change_points(country_events_df, "Country", pen=80, min_size=8))
cp_list.append(detect_change_points(region_events_df, "Region", pen=80, min_size=8))

pd.set_option("display.max_rows", None)
cp_table = pd.concat(cp_list, ignore_index=True)
cp_table

,Country,Region,level,change_point_date
0,Australia,None,Country,2021-05-30
1,Australia,None,Country,2021-10-17
2,Australia,None,Country,2022-01-30
3,Austria,None,Country,2018-01-21
4,Austria,None,Country,2018-04-01
5,Austria,None,Country,2018-08-19
6,Austria,None,Country,2019-01-06
7,Austria,None,Country,2019-11-17
8,Austria,None,Country,2020-07-19
9,Austria,None,Country,2021-05-30


Plotting the daily median distance to Zurich of all observations over time, as well as annual plots to see it better

In [11]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])
df["date"] = pd.to_datetime(df["date"])

zurich = (47.3769, 8.5417)

df["distance_zurich"] = df.apply(
    lambda row: haversine(
        zurich,
        (row["latitude"], row["longitude"]),
        unit="km"
    ),
    axis=1
)

daily_distance = (
    df
    .groupby("date")["distance_zurich"]
    .median()
    .reset_index()
)

daily_distance["distance_30d_mean"] = (
    daily_distance["distance_zurich"]
    .rolling(window=30, min_periods=1)
    .mean()
)

plt.figure(figsize=(16,8))

plt.plot(
    daily_distance["date"],
    daily_distance["distance_zurich"],
    alpha=0.6,
    label="Daily median"
)

plt.plot(
    daily_distance["date"],
    daily_distance["distance_30d_mean"],
    linewidth=2,
    label="30-day running mean"
)

plt.ylabel("Distance [km]", fontsize=12)
plt.xlabel("Date", fontsize=12)
plt.title("Daily Median Distance of Observations to Zurich over time", fontsize=15)
plt.legend()
plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.4)
plt.ylim(0,8000)
plt.savefig("../Products/Distance_to_Zurich/Median_Distance_to_Zurich_full.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_3488\372182383.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 42

annual distance-to-Zurich plots

In [12]:
# uses daily_distance

daily_distance["year"] = daily_distance["date"].dt.year
years = sorted(daily_distance["year"].unique())

for year in years:

    data = daily_distance[daily_distance["year"] == year]
    year_median = df[df["date"].dt.year == year]["distance_zurich"].median()

    plt.figure(figsize=(16,8))

    plt.plot(
        data["date"],
        data["distance_zurich"],
        alpha=0.6,
        label="Daily median"
    )

    plt.plot(
        data["date"],
        data["distance_30d_mean"],
        linewidth=2,
        label="30-day running mean"
    )

    plt.text(
        0.02,
        0.95,
        f"Yearly median distance: {year_median:.0f} km",
        transform=plt.gca().transAxes,
        fontsize=12,
        verticalalignment="top"
    )

    plt.ylabel("Distance [km]", fontsize=12)
    plt.xlabel("Date", fontsize=12)
    plt.title(f"Daily Median Distance of Observations to Zurich over time - {year}", fontsize=15)
    plt.legend(loc="upper right")
    plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.4)
    plt.ylim(0,8000)
    plt.savefig(f"../Products/Distance_to_Zurich/Median_Distance_to_Zurich_{year}.png", dpi=300, bbox_inches="tight")
    plt.close()